In [1]:
import matplotlib.pyplot as plt
import pandas as pd

from processL0b.reader import Reader


r = Reader('C:\\Users\\adamgw\\Desktop\\data413\\l0b\\26_04_13__17_37_08__LN2Nohyms.h5')

c:\Users\adamgw\Desktop\Code\DataSystemPy3\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


For each radiometer row, we want to associate a temperature row and a gps row.
Find the closest system timestamp and tie those together.

In [2]:
# This gets the radiometer rows between the ith and ith + 1 thermistor timestamp
# Each step contains about 5500 rows, but some rows contain double or even triple? Why is that?
# for i in range(len(r.thermistors.data) - 1):
i = 3
subset = r.radiometer.data[
    (r.radiometer.data['Timestamp'] - r.thermistors.temps['Timestamp'][i] > 0)
    &
    (r.radiometer.data['Timestamp'] - r.thermistors.temps['Timestamp'][i+1] < 0)
]

# Filter this to the calibration target
subset = subset[
    (subset['MotorPosition'] > 3000)
    &
    (subset['MotorPosition'] < 5000)
]

# Get the calibration target temperature for this timestamp
caltar_thermistors = [9,10,11,12,13,14,15,16]
caltar_temp = r.thermistors.temps[caltar_thermistors].mean(axis=1).iloc[i]


# Average the counts per channel and get the data point (counts, temperature)
(caltar_temp, subset.mean())

(np.float64(262.10201968589524),
 Timestamp          1.776123e+09
 MotorPosition      4.000760e+03
 34 QV              9.691497e+03
 Not Connected 1    1.623692e+04
 18 QV              7.246404e+03
 24 QV              9.461713e+03
 34 QH              1.083709e+04
 Not Connected 2    1.631132e+04
 18 QH              6.420321e+03
 24 QH              8.142479e+03
 Revolution         3.000000e+00
 dtype: float64)

In [3]:
r.get_calibration_point(3, caltar_thermistors, 3000, 4000)

(np.float64(262.10201968589524),
 Timestamp          1.776123e+09
 MotorPosition      3.500821e+03
 34 QV              9.750788e+03
 Not Connected 1    1.623680e+04
 18 QV              7.368298e+03
 24 QV              9.558689e+03
 34 QH              1.087528e+04
 Not Connected 2    1.631132e+04
 18 QH              6.559843e+03
 24 QH              8.269907e+03
 Revolution         3.000000e+00
 dtype: float64)